# Подготовка трекера - Desktop-путь (BoT-SORT + OSNet ReID)

Путь для desktop GPU: детектор PyTorch/ONNX (CUDA) + **BoT-SORT + OSNet ReID** (`boxmot`).
ReID-внешность сшивает ID через окклюзии в плотной сцене (нужен torch). Параметры
BoT-SORT синхронизированы с `app/config.py` (`OSNET_*`);

In [ ]:
import json
from pathlib import Path
import cv2
import numpy as np

if not hasattr(np, "asfarray"):          
    np.asfarray = lambda a, dtype=float: np.asarray(a, dtype=dtype)

import pandas as pd
import torch
import motmetrics as mm
from boxmot import BotSort
from ultralytics import YOLO
from tqdm.notebook import tqdm

ROOT = Path().resolve().parent
MOT20_DIR = ROOT / "data" / "MOT20" / "MOT20" / "train"
MODELS_DIR = ROOT / "models"
MODEL_WEIGHTS = MODELS_DIR / "yolo11n_mot20_v2.pt"
REID_WEIGHTS = MODELS_DIR / "osnet_x0_25_msmt17.pt"

DET_CONF, DET_IOU, DET_IMGSZ, DEVICE, FRAME_RATE = 0.25, 0.45, 640, "cuda", 25
EVAL_SEQUENCES = ["MOT20-01", "MOT20-02", "MOT20-03", "MOT20-05"]

# Параметры BoT-SORT = OSNET_* из app/config.py
BOTSORT = dict(track_high_thresh=0.30, new_track_thresh=0.50, track_buffer=90, match_thresh=0.80, appearance_thresh=0.25, cmc_method="ecc")

assert MODEL_WEIGHTS.exists(), f"Нет весов {MODEL_WEIGHTS} — запусти detector_training.ipynb"
assert REID_WEIGHTS.exists(), f"Нет ReID-весов {REID_WEIGHTS} (osnet_x0_25_msmt17)"
MODEL_WEIGHTS

In [ ]:
def load_mot_gt(gt_path):
    """GT MOT20 -> {frame_id: [(track_id, x, y, w, h), ...]}"""
    gt = {}
    for line in gt_path.read_text().strip().splitlines():
        p = line.split(",")
        if int(p[6]) == 0 or int(p[7]) != 1 or (len(p) > 8 and float(p[8]) < 0.25):
            continue
        gt.setdefault(int(p[0]), []).append((int(p[1]), float(p[2]), float(p[3]), float(p[4]), float(p[5])))
    return gt


def mot_metrics(gt, pred):
    """IDF1 / MOTA / IDSW / Precision / Recall через motmetrics"""
    acc = mm.MOTAccumulator(auto_id=True)
    for fid in sorted(set(gt) | set(pred)):
        g, h = gt.get(fid, []), pred.get(fid, [])
        gb = np.array([[o[1], o[2], o[3], o[4]] for o in g]) if g else np.empty((0, 4))
        hb = np.array([[o[1], o[2], o[3], o[4]] for o in h]) if h else np.empty((0, 4))
        acc.update([o[0] for o in g], [o[0] for o in h], mm.distances.iou_matrix(gb, hb, max_iou=0.5))
    s = mm.metrics.create().compute(acc, metrics=["idf1", "mota", "motp", "num_switches", "precision", "recall"], name="e")
    return {"idf1": float(s["idf1"].iloc[0]), "mota": float(s["mota"].iloc[0]), "idsw": int(s["num_switches"].iloc[0]), "prec": float(s["precision"].iloc[0]), "rec": float(s["recall"].iloc[0])}


def run_botsort(model, seq_dir):
    """Детектор + BoT-SORT+OSNet по кадрам -> {frame_id: [(track_id, x, y, w, h), ...]}"""
    trk = BotSort(reid_weights=REID_WEIGHTS, device=torch.device("cuda:0"), half=False, frame_rate=FRAME_RATE, with_reid=True, **BOTSORT)
    preds = {}
    for ff in sorted((seq_dir / "img1").glob("*.jpg")):
        frame = cv2.imread(str(ff))
        r = model.predict(source=frame, imgsz=DET_IMGSZ, conf=DET_CONF, iou=DET_IOU, device=DEVICE, verbose=False, classes=[0])[0]
        if r.boxes is not None and len(r.boxes):
            arr = np.concatenate([r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()[:, None], np.zeros((len(r.boxes), 1))], axis=1).astype(np.float32)
        else:
            arr = np.empty((0, 6), dtype=np.float32)
        res = trk.update(arr, frame)
        preds[int(ff.stem)] = [(int(o[4]), float(o[0]), float(o[1]), float(o[2] - o[0]), float(o[3] - o[1])) for o in res]
    return preds


model = YOLO(str(MODEL_WEIGHTS)).to(DEVICE)

## IDF1 BoT-SORT + OSNet на MOT20

In [ ]:
rows = []
for seq in tqdm(EVAL_SEQUENCES, desc="botsort"):
    s = MOT20_DIR / seq
    rows.append({"sequence": seq, **mot_metrics(load_mot_gt(s / "gt" / "gt.txt"), run_botsort(model, s))})

df_bs = pd.DataFrame(rows)
df_bs

In [ ]:
params = {"tracker": "botsort_osnet_msmt17", **BOTSORT, "with_reid": True,
          "reid_weights": REID_WEIGHTS.name, "model_weights": MODEL_WEIGHTS.name,
          "idf1_mot20_train": round(float(df_bs["idf1"].mean()), 4),
          "mota_mot20_train": round(float(df_bs["mota"].mean()), 4),
          "per_seq": {r["sequence"]: round(r["idf1"], 4) for _, r in df_bs.iterrows()}}
out = MODELS_DIR / "botsort_mot20_idf1.json"
out.write_text(json.dumps(params, indent=2, ensure_ascii=False))
params